`ingest_images`

Script examples to download per-building imagery from Google Satellite and Google Street View

**Credential management (e.g. StreetView)**

API keys and service passwords live in a single YAML file alongside `config.yaml`:

```
%APPDATA%\placeslab\openplaces\credentials.yaml   # Windows
~/.config/openplaces/credentials.yaml             # Linux / macOS
```

One entry per service, keyed by scraper or source ID:

```yaml
# openplaces credentials (keep in secure location)

google_streetview:
  api_key: "YOUR_KEY_HERE"

# USGS: Landsat metadata
# usgs:
#   username: ""
#   password: ""
```

In [ ]:
# Print file path and which services are registered, without revealing the values
from openplaces.config import show_credentials

show_credentials()

# Configure

In [ ]:
import argparse

from openplaces.io.ingester import Ingester

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(description='Ingest building images using a recipe')
parser.add_argument(
    '--recipe_id',
    help='Identifier of the recipe (e.g., "image-googlesatellite-z20")',
)
parser.add_argument(
    '--admin_ids',
    help='Administrative unit IDs to ingest (e.g., "US-MA-MI-SO")',
    nargs='*',
)
parser.add_argument(
    '--target_recipe_id',
    help=(
        'Recipe ID of the harmonized entity to photograph '
        '(overrides entity_recipe in the image recipe YAML). '
        'E.g. "US_building-nsi-2026" for NSI point buildings, '
        '"US_footprint-cheer-2026" for CHEER polygon footprints.'
    ),
    default=None,
)
parser.add_argument(
    '--n_sample',
    help='Cap the number of buildings per admin unit (for test runs)',
    type=int,
    default=None,
)
parser.add_argument(
    '--reprocess',
    help='Re-fetch images even if output metadata parquet already exists',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    help='If True, print outputs while processing data',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    # Google Satellite (no credentials required)
    # '--recipe_id image-googlesatellite-z20 '
    # Google Street View (requires 'google_streetview' entry in credentials.yaml)
    '--recipe_id image-googlestreetview-2026 '
    #
    # Somerville, MA
    '--admin_ids US-MA-MI-SO '
    # Arlington, MA
    # '--admin_ids US-MA-MI-AR '
    # Cambridge, MA
    # '--admin_ids US-MA-MI-CA '
    #
    # Override the entity recipe (omit to use the YAML default)
    # '--target_recipe_id US_building-nsi-2026 '  # NSI point buildings
    # '--target_recipe_id US_footprint-cheer-2026 ' # CHEER polygon footprints
    #
    # Sample size for test runs (comment out to ingest all buildings)
    '--n_sample 10 '
    #
    # '--reprocess '
    '--verbose '
)

args_list = [x for x in ARGS_TEST.split(' ') if x != '']
args = parser.parse_args(args_list)
args

In [ ]:
from openplaces.recipe import get_recipe_by_id
from openplaces.utils import pretty_print

pretty_print(get_recipe_by_id(args.recipe_id))

# Ingest images

In [ ]:
ingester = Ingester(args.recipe_id, args.admin_ids, verbose=args.verbose)
if args.n_sample is not None:
    ingester.recipe['n_sample'] = args.n_sample

In [ ]:
# ingester.ingest(reprocess=args.reprocess, target_recipe_id=args.target_recipe_id)

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/'
# If False, writes a test version of the script to 'scripts/_test/'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Inspect results

## Metadata parquet

In [ ]:
import openplaces as op

recipe = get_recipe_by_id(args.recipe_id)
meta = op.get_entities(recipe, args.admin_ids[0])
print(f'{len(meta):,d} images')
meta.head()

## Footprint overlay

In [ ]:
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import rasterio
from pyproj import Transformer

from openplaces.core.schema import AdminId

entity_recipe = get_recipe_by_id(recipe['entity_recipe'])
fp_admin_id = AdminId(*AdminId(args.admin_ids[0]).levels[:3])
footprints_all = op.get_entities(entity_recipe, str(fp_admin_id), geom=True)
footprints = footprints_all.loc[footprints_all.index.isin(meta.index)].to_crs(
    'EPSG:4326'
)

scraper = recipe.get('image_scraper', '')
sample_ids = meta.index[: min(3, len(meta))]
n = len(sample_ids)

fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
if n == 1:
    axes = [axes]

for ax, fid in zip(axes, sample_ids):
    p = Path(meta.loc[fid, 'image_path'])
    geom = footprints.loc[fid, 'geometry'] if fid in footprints.index else None
    cx = geom.centroid.x if geom is not None else 0.0
    cy = geom.centroid.y if geom is not None else 0.0

    if scraper == 'google_satellite' and p.exists():
        # Project image bounds and footprint into a local orthographic CRS
        # centred on this building (axes in metres, equal physical scale).
        to_ortho = Transformer.from_crs(
            'EPSG:4326',
            f'+proj=ortho +lat_0={cy} +lon_0={cx} +datum=WGS84',
            always_xy=True,
        )
        with rasterio.open(p) as src:
            b = src.bounds
            img_arr = src.read().transpose(1, 2, 0)
        left_m, bottom_m = to_ortho.transform(b.left, b.bottom)
        right_m, top_m = to_ortho.transform(b.right, b.top)
        ax.imshow(img_arr, extent=[left_m, right_m, bottom_m, top_m], origin='upper')
        ax.set_aspect('equal')
        if geom is not None:
            x_m, y_m = to_ortho.transform(
                list(geom.exterior.xy[0]), list(geom.exterior.xy[1])
            )
            ax.plot(x_m, y_m, color='red', linewidth=1.5)
        ax.set_xlabel('m')
        ax.set_ylabel('m')
    else:
        if p.exists():
            ax.imshow(mpimg.imread(p))

    ax.set_title(str(fid)[:24], fontsize=8)

source_label = 'Satellite' if scraper == 'google_satellite' else 'Street View'
source_label = 'Satellite' if scraper == 'google_satellite' else 'Street View'
plt.tight_layout()
plt.show()

## Story count detection

In [ ]:
from openplaces.io.enricher.detectors.n_stories import NStoriesDetector
from openplaces.io.scrapers.types import Image, ImageSet

image_set = ImageSet()
for idx, row in meta.iterrows():
    p = Path(row['image_path'])
    if p.exists():
        image_set.dir_path = str(p.parent)
        image_set.add_image(idx, Image(p.name))

print(f'Estimating story counts for {len(image_set.images)} images...')
predictions = NStoriesDetector().predict(image_set)

n_stories = meta[['image_path']].assign(n_stories_est=meta.index.map(predictions))
n_stories

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

sample_ids = n_stories.index[: min(9, len(n_stories))]
nrows, ncols = 3, 3

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = axes.flatten()

for ax, fid in zip(axes, sample_ids):
    p = Path(n_stories.loc[fid, 'image_path'])
    count = n_stories.loc[fid, 'n_stories_est']
    label = (
        f'{count} stor{"y" if count == 1 else "ies"}'
        if count is not None
        else 'no detection'
    )
    if p.exists():
        ax.imshow(mpimg.imread(p))
    ax.set_title(f'{str(fid)[:20]}\n{label}', fontsize=8)
    ax.axis('off')

for ax in axes[len(sample_ids) :]:
    ax.axis('off')

plt.tight_layout()
plt.show()